In [1]:
# Food Delivery Data Analysis  
### Innomatics Research Labs – Advanced GenAI Internship Entrance Test




In [2]:
import pandas as pd
import sqlite3


In [3]:
# Load CSV and JSON files
orders = pd.read_csv("orders.csv")
users = pd.read_json("users.json")

# Load SQL file into in-memory SQLite database
conn = sqlite3.connect(":memory:")
with open("restaurants.sql", "r", encoding="utf-8") as f:
    conn.executescript(f.read())

restaurants = pd.read_sql("SELECT * FROM restaurants", conn)


In [4]:
df = orders.merge(users, on="user_id", how="left") \
           .merge(restaurants, on="restaurant_id", how="left")

df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [5]:
df[df["membership"]=="Gold"] \
.groupby("city")["total_amount"] \
.sum() \
.sort_values(ascending=False)


city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [6]:
df.groupby("cuisine")["total_amount"] \
.mean() \
.sort_values(ascending=False)


cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [7]:
df.groupby("user_id")["total_amount"] \
.sum() \
.reset_index() \
.query("total_amount > 1000")["user_id"] \
.nunique()


2544

In [8]:
df["rating_range"] = pd.cut(
    df["rating"],
    bins=[0,3.5,4.0,4.5,5.0],
    labels=["3.0–3.5","3.6–4.0","4.1–4.5","4.6–5.0"]
)

df.groupby("rating_range")["total_amount"] \
.sum() \
.sort_values(ascending=False)


C:\Users\GIRISH PANCHARIYA\AppData\Local\Temp\ipykernel_20212\1096150018.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("rating_range")["total_amount"] \


rating_range
4.6–5.0    2197030.75
3.0–3.5    2136772.70
4.1–4.5    1960326.26
3.6–4.0    1717494.41
Name: total_amount, dtype: float64

In [9]:
df[df["membership"]=="Gold"] \
.groupby("city")["total_amount"] \
.mean() \
.sort_values(ascending=False)


city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [10]:
df.groupby("cuisine").agg(
    distinct_restaurants=("restaurant_id","nunique"),
    revenue=("total_amount","sum")
).sort_values("distinct_restaurants")


,distinct_restaurants,revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [11]:
round(
    (df[df["membership"]=="Gold"].shape[0] / df.shape[0]) * 100
)


50

In [12]:
df.groupby("name").agg(
    orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
).query("orders < 20") \
 .sort_values("avg_order_value", ascending=False)


,orders,avg_order_value
name,,
User_2429,1,1497.42
User_889,1,1492.63
User_1843,1,1484.24
User_1882,1,1481.84
User_925,1,1476.18
...,...,...
User_2368,1,119.08
User_2583,1,117.62
User_1188,1,109.58


In [13]:
df.groupby(["membership","cuisine"])["total_amount"] \
.sum() \
.sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [14]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["quarter"] = df["order_date"].dt.to_period("Q")

df.groupby("quarter")["total_amount"] \
.sum() \
.sort_values(ascending=False)


C:\Users\GIRISH PANCHARIYA\AppData\Local\Temp\ipykernel_20212\3264237451.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["order_date"] = pd.to_datetime(df["order_date"])


quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

In [15]:
print("Total Gold Orders:", df[df["membership"]=="Gold"].shape[0])
print("Hyderabad Revenue:", round(df[df["city"]=="Hyderabad"]["total_amount"].sum()))
print("Distinct Users:", df["user_id"].nunique())
print("Avg Gold Order Value:", round(df[df["membership"]=="Gold"]["total_amount"].mean(),2))
print("Orders with Rating ≥ 4.5:", df[df["rating"]>=4.5].shape[0])

top_gold_city = df[df["membership"]=="Gold"] \
.groupby("city")["total_amount"] \
.sum() \
.idxmax()

print("Orders in Top Gold City:", 
      df[(df["membership"]=="Gold") & (df["city"]==top_gold_city)].shape[0])


Total Gold Orders: 4987
Hyderabad Revenue: 1889367
Distinct Users: 2883
Avg Gold Order Value: 797.15
Orders with Rating ≥ 4.5: 3374
Orders in Top Gold City: 1337
